# <font color='green'>Required Libary Install</font>


In [ ]:
!pip install scikit-learn 

In [ ]:
!pip install tensorflow

In [ ]:
!pip install matplotlib

In [ ]:
!pip install plotly


In [ ]:
!pip install seaborn

In [ ]:
!pip install imblearn 

In [ ]:
!pip install xgboost

# <font color='Red'>Required Libary Import</font>


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

from collections import Counter as C
import plotly.graph_objects as go


In [ ]:
import plotly.subplots as sp
from scipy.stats import gaussian_kde

import seaborn as sns
import matplotlib.pyplot as plt

from imblearn.over_sampling import SMOTE

import plotly.graph_objects as go
import plotly.express as px

from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, accuracy_score
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from plotly.subplots import make_subplots

import joblib

# <font color='Blue'>Load the Dataset</font>


In [ ]:
# df = pd.read_excel("assignment.xlsx", sheet_name="assignment", engine='openpyxl')
df = pd.read_csv("assignment.csv")  


### <font color='Green'>Understanding Data</font>  
**Goal**: Understand the structure and size of the dataset.  

- Check the number of rows and columns.  
- Examine column names and data types.  
- Identify missing values and duplicate rows.

In [ ]:


# Check the number of rows and columns
print("Shape of the dataset:", df.shape)
print("-"*100)

# Examine column names and data types
print("\nColumn names and data types:")
print(df.dtypes)
print("-"*100)

# Identify missing values
print("\nMissing values per column:")
print(df.isnull().sum())
print("-"*100)

# Check for duplicate rows
duplicate_rows = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicate_rows}")

print("-"*100)

# Check for duplicates
print(df.duplicated().sum())

In [ ]:
# View the first 5 rows of the dataset
print("First 5 rows:")
df.head()


In [ ]:
# View the last 5 rows of the dataset
print("\nLast 5 rows:")
df.tail()

In [ ]:
# Set columns name
df.columns = ['feature1', 'feature2', 'feature3', 'feature4', 'feature5', 'feature6', 'feature7', 'target_cols']


In [ ]:
# View the first 5 rows of the dataset
print("First 5 rows:")
df.head()

In [ ]:
# Summary statistics for all numerical features
summary_stats = df.describe().T  # Transpose for better readability
summary_stats['median'] = df.median()  # Add median
summary_stats['missing_values'] = df.isnull().sum()
summary_stats['skewness'] = df.skew()

# Sort by skewness to detect highly skewed features
summary_stats_sorted = summary_stats.sort_values(by='skewness', ascending=False)

# Display summary
summary_stats_sorted[['mean', 'median', 'std', 'min', '25%', '50%', '75%', 'max', 'skewness', 'missing_values']]


# <font color='Blue'>Target Variable Analysis : target_cols </font>
Investigate the nature of the target variable.

Determine if the dataset is imbalanced.

Understand the distribution of classes (e.g., 0 vs. 1).

Visualize class frequencies.

In binary classification tasks where class imbalance can significantly affect model performance.

In [ ]:
print(df['target_cols'].value_counts())
print("#"*100)
print(df['target_cols'].value_counts(normalize=True) * 100)

In [ ]:


def plot_binary_target_distribution_plotly(data, target_col):
    """
    Plots the distribution of a binary target column using a Plotly bar chart.

    Parameters:
    - data (pd.DataFrame): The DataFrame containing the target column.
    - target_col (str): The name of the binary target column.
    """
    # Count values
    value_counts = data[target_col].value_counts().sort_index()
    labels = value_counts.index.astype(str)
    counts = value_counts.values

    # Create bar chart
    fig = go.Figure(data=[
        go.Bar(x=labels, y=counts, text=counts, textposition='auto', marker_color=['skyblue', 'salmon'])
    ])

    # Update layout with size
    fig.update_layout(
        title=f'Distribution of Binary Target Column: {target_col}',
        xaxis_title='Class',
        yaxis_title='Count',
        template='plotly_white',
        width=600,   # Width in pixels
        height=400   # Height in pixels
    )

    fig.show()


In [ ]:
plot_binary_target_distribution_plotly(df, 'target_cols')


#### <font color='Red'> Note : Dataset is imbalance</font>
target_cols : total of records

0.0    277664

1.0    139469

####################################################################################################

total no of records in percentage

0.0  :  66.564861 %

1.0  :  33.435139 %


# <font color='Green'> Feature Distributions Analysis</font>
Visualize how individual features are distributed.

Use histograms and KDE plots.

Analyze how each feature behaves across different target classes.

understanding skewness, spread, and modality of each feature.

In [ ]:

def plot_feature_histograms_plotly(data, bins=30, width=1200, height=1000):
    """
    Plots histograms for each numeric feature in a DataFrame using Plotly subplots.

    Parameters:
    - data (pd.DataFrame): The input DataFrame.
    - bins (int): Number of histogram bins.
    - width (int): Width of the overall figure.
    - height (int): Height of the overall figure.
    """
    numeric_cols = data.select_dtypes(include='number').columns
    num_features = len(numeric_cols)
    cols = 3  # number of subplot columns
    rows = -(-num_features // cols)  # ceiling division for rows

    fig = sp.make_subplots(rows=rows, cols=cols, subplot_titles=numeric_cols)

    for i, col in enumerate(numeric_cols):
        row = i // cols + 1
        col_pos = i % cols + 1
        fig.add_trace(
            go.Histogram(x=data[col], nbinsx=bins, name=col),
            row=row, col=col_pos
        )

    fig.update_layout(
        title="Histograms for Each Feature",
        showlegend=False,
        height=height,
        width=width,
        template='plotly_white'
    )
    fig.show()


In [ ]:
df_features = df.iloc[:,:-1]
plot_feature_histograms_plotly(df_features)


In [ ]:


def plot_kde_by_class_plotly(data, target_col, width=900, height=400):
    """
    Plots KDE-like distributions for each feature by target class using Plotly.

    Parameters:
    - data (pd.DataFrame): DataFrame containing features and target column.
    - target_col (str): Name of the binary/class target column.
    - width (int): Width of the overall figure 
    - height (int): Height of the overall figure 
    """
    feature_cols = data.columns.difference([target_col])
    
    for col in feature_cols:
        # Create an empty figure
        fig = go.Figure()

        # For each target class, compute the KDE
        for target in data[target_col].unique():
            class_data = data[data[target_col] == target][col]
            kde = gaussian_kde(class_data)
            x_values = np.linspace(class_data.min(), class_data.max(), 100)
            y_values = kde(x_values)

            fig.add_trace(go.Scatter(
                x=x_values, 
                y=y_values, 
                mode='lines', 
                name=f'Class {target}', 
                fill='tozeroy' if target == data[target_col].min() else None
            ))

        # Update layout with custom width and height
        fig.update_layout(
            title=f'Distribution of {col} by Target Class',
            xaxis_title=col,
            yaxis_title='Density',
            template='plotly_white',
            showlegend=True,
            width=width,    # Set the width
            height=height   # Set the height
        )
        
        # Show the figure
        fig.show()


In [ ]:
plot_kde_by_class_plotly(df, target_col='target_cols')


# <font color='Green'> Correlation Analysis</font>
Identify linear relationships between features.

Use a heatmap to spot multicollinearity.

Check feature-to-target correlation to find useful predictors.

In [ ]:

def plot_correlation_matrix_plotly(data, target_col):
    # Compute the correlation matrix
    corr_matrix = data.corr()

    # Create the heatmap for the correlation matrix
    fig = go.Figure(data=go.Heatmap(
        z=corr_matrix.values,
        x=corr_matrix.columns,
        y=corr_matrix.index,
        colorscale='RdBu',  # Updated to a valid colorscale
        zmin=-1, zmax=1,  # Scale from -1 to 1
        colorbar=dict(title="Correlation", tickvals=[-1, 0, 1]),
        text=corr_matrix.values,  # Show the correlation values
        hovertemplate='%{text}'
    ))

    fig.update_layout(
        title="Feature Correlation Matrix",
        xaxis_title="Features",
        yaxis_title="Features",
        template='plotly_white',
        width=800,
        height=600
    )
    fig.show()

    # Correlation of features with the target column
    target_corr = corr_matrix[target_col].sort_values(ascending=False)

    # Plot the correlation of features with the target column
    bar_fig = px.bar(
        target_corr,
        x=target_corr.index,
        y=target_corr.values,
        labels={'x': 'Features', 'y': 'Correlation with ' + target_col},
        title=f'Correlation of Features with {target_col}',
        color=target_corr.values,
        color_continuous_scale='Viridis'  # Valid continuous color scale for the bars
    )
    bar_fig.update_layout(
        template='plotly_white',
        width=800,
        height=500
    )
    bar_fig.show()

    # Optionally print correlation of features with target
    print(target_corr)


In [ ]:
plot_correlation_matrix_plotly(df, target_col='target_cols')


# <font color='pinkblue'> Outlier Detection</font>
Identify extreme values that may distort the model.

Use boxplots for each feature.

Helps determine if outlier handling (removal or transformation) is necessary.

In [ ]:
for col in df.columns[:-1]:
    sns.boxplot(x='target_cols', y=col, data=df)
    plt.title(f'{col} Distribution by Class')
    plt.show()

# <font color='Red'> Feature Relationships</font>
Goal: Understand interactions between multiple features.

Use pairplot() to see feature combinations and their separability across target classes.

In [ ]:
sns.pairplot(df.sample(1000), hue='target_cols') 
plt.show()


In [ ]:
# <font color='Green'>Feature and Target Separation</font>

# <font color='Yelloblack'>Feature and Target Selection</font>

In [ ]:
X = df.iloc[:, :-1].values 
y = df.iloc[:, -1].values  

In [ ]:
#checking Imbalance 
print(C(y))

# <font color='Green'>Handling Imbalanced Datasets Problem</font>

In [ ]:
print(df['target_cols'].value_counts())
print("#"*100)
print(df['target_cols'].value_counts(normalize=True) * 100)

In [ ]:

# Handle class imbalance using SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)


In [ ]:
#checking balance 
print(C(y_resampled))
print(len(X_resampled))

# <font color='red'>Train-Test Split</font>
dataset into training and testing subsets. use 80% for training and 20% for testing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# <font color='yellowred'>Feature Scaling</font>
Always fit the scaler on the training set only and then apply it to the test set. This prevents data leakage and ensures proper generalization.

In [ ]:
# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Determine the number of unique target classes
num_classes = len(np.unique(y_train))
print(num_classes)

# <font color='Green'>Define the Embedding Model</font>
Deep neural network that reduces the high-dimensional feature input into a 2D embedding.

In [ ]:
# Define the embedding model
embedding_dim = 2

# Define the input shape based on number of features
input_shape = (X_train_scaled.shape[1],)

In [ ]:
embedding_model = keras.Sequential([
    layers.Input(shape=input_shape),  # Input layer

    # First hidden layer without BatchNormalization and Dropout
    layers.Dense(128, activation='relu'),

    # Second hidden layer with BatchNormalization and Dropout
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    # Third hidden layer with BatchNormalization and Dropout
    layers.Dense(32, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    # Output layer: 2D embedding
    layers.Dense(embedding_dim)
])


# <font color='pink'>Define Classification Model</font>
classification layer on top of the embedding model. The model uses the previously defined embedding as a feature extractor, and then a softmax output layer is used for class prediction.

In [ ]:
# Define the classification model on top of the embedding model
classification_model = keras.Sequential([
    embedding_model,  # Pre-trained embedding model
    layers.Dense(1, activation='sigmoid')  # Classification layer
])

# <font color='blue'>Compile the Binary Classification Model</font>

In [ ]:
# Compile the binary classification model
classification_model.compile(
    optimizer='adam',  # Adam optimizer for adaptive learning rate
    loss='binary_crossentropy',  # Loss function for binary classification
    metrics=['accuracy']  # Evaluate accuracy during training
)

# <font color='Green'>Train the Model</font>
epochs: The number of times the entire training dataset is passed through the model. You can increase this for better performance but risk overfitting if it's too high.

batch_size: The number of samples processed before the model is updated. A smaller batch size might make training slower, but it can help the model generalize better.

validation_data: The validation data (X_test_scaled and y_test) allows us to monitor the model's performance on unseen data during training.

In [ ]:
# Define training parameters
epochs = 40  # Number of epochs to train # used 3 epoach because it is taking time
batch_size = 32  # Batch size

# Train the classification model
history = classification_model.fit(
    X_train_scaled, y_train,  # Training data
    epochs=epochs,  # Number of epochs
    batch_size=batch_size,  # Batch size
    validation_data=(X_test_scaled, y_test),  # Validation data
    verbose=1  # Display progress
)


# <font color='Green'>Analysis Training Progress</font>

In [ ]:

def plot_training_history_plotly(history):
    """
    Plots training and validation accuracy and loss using Plotly.

    Parameters:
    - history: Keras History object returned by model.fit()
    """
    # Plot Accuracy
    fig_acc = go.Figure()
    fig_acc.add_trace(go.Scatter(
        y=history.history['accuracy'], mode='lines+markers', name='Train Accuracy'))
    fig_acc.add_trace(go.Scatter(
        y=history.history['val_accuracy'], mode='lines+markers', name='Val Accuracy'))

    fig_acc.update_layout(
        title='Model Accuracy',
        xaxis_title='Epoch',
        yaxis_title='Accuracy',
        template='plotly_white',
        width=800,
        height=500
    )
    fig_acc.show()

    # Plot Loss
    fig_loss = go.Figure()
    fig_loss.add_trace(go.Scatter(
        y=history.history['loss'], mode='lines+markers', name='Train Loss'))
    fig_loss.add_trace(go.Scatter(
        y=history.history['val_loss'], mode='lines+markers', name='Val Loss'))

    fig_loss.update_layout(
        title='Model Loss',
        xaxis_title='Epoch',
        yaxis_title='Loss',
        template='plotly_white',
        width=800,
        height=500
    )
    fig_loss.show()


In [ ]:
plot_training_history_plotly(history)


# <font color='red'>Model Evaluation and Predicting Binary & Probabilities </font>

In [ ]:
loss, accuracy = classification_model.evaluate(X_test_scaled, y_test)
print(f"Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")

In [ ]:
y_pred_prob = classification_model.predict(X_test_scaled).flatten()
y_pred_binary = (y_pred_prob > 0.5).astype(int)
y_pred_binary

In [ ]:

# Evaluate predictions
y_pred_prob = classification_model.predict(X_test_scaled).flatten()
y_pred_binary = (y_pred_prob > 0.5).astype(int)

def evaluate_classification(y_true, y_pred_binary, y_pred_prob):
    precision = precision_score(y_true, y_pred_binary)
    recall    = recall_score(y_true, y_pred_binary)
    f1        = f1_score(y_true, y_pred_binary)
    roc_auc   = roc_auc_score(y_true, y_pred_prob)
    accuracy  = accuracy_score(y_true, y_pred_binary)    

    print(f'Precision : {precision:.2f}')
    print(f'Recall    : {recall:.2f}')
    print(f'F1-Score  : {f1:.2f}')
    print(f'ROC AUC   : {roc_auc:.2f}')
    print(f'Model Accuracy   : {accuracy:.2f}')

    return {
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'ROC AUC': roc_auc,
        'Model Accuracy' : accuracy
    }

metrics = evaluate_classification(y_test, y_pred_binary, y_pred_prob)

def plot_metrics_bar(metrics_dict, width=900, height=400):
    custom_colors = ['#FF6347', '#4682B4', '#32CD32', '#FFD700', '#8A2BE2']  # Tomato, SteelBlue, LimeGreen, Gold, BlueViolet
    metric_names = list(metrics_dict.keys())
    metric_values = list(metrics_dict.values())

    fig = go.Figure()

    for i, (name, value) in enumerate(zip(metric_names, metric_values)):
        fig.add_trace(go.Bar(
            x=[name],
            y=[value],
            name=name,
            marker_color=custom_colors[i % len(custom_colors)],
            text=f'{value:.2f}',
            textposition='outside'
        ))

    fig.update_layout(
        title='Evaluation Metrics',
        xaxis_title='Metric',
        yaxis_title='Score',
        template='plotly_white',
        width=width,
        height=height,
        showlegend=True
    )

    fig.show()

plot_metrics_bar(metrics, width=800, height = 500)




In [ ]:
confusion_matrix(y_test, y_pred_binary)

# <font color='violet'> Model Training with Callbacks </font>

In [ ]:

# Define callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3)

# Train the model
history = classification_model.fit(
    X_train_scaled, y_train,  # Training data
    epochs=50,
    batch_size=32,
    validation_data=(X_test_scaled, y_test),
    callbacks=[early_stop, reduce_lr],
    verbose=1
)


In [ ]:
from plotly.subplots import make_subplots
# Evaluate the model
y_pred = classification_model.predict(X_test_scaled)
y_pred_binary = (y_pred > 0.5).astype(int)

# Print the classification report
classification_report_str = classification_report(y_test, y_pred_binary)
print(classification_report_str)

# ROC AUC score
auc_score = roc_auc_score(y_test, y_pred)
print(f'AUC: {auc_score:.2f}')

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_binary)

# Plot Confusion Matrix using Plotly Heatmap
fig_cm = go.Figure(data=go.Heatmap(
    z=cm,
    x=['Negative', 'Positive'],
    y=['Negative', 'Positive'],
    colorscale='Blues',
    zmin=0, zmax=np.max(cm),
    text=cm,
    hoverinfo='text',
))

fig_cm.update_layout(
    title='Confusion Matrix',
    xaxis_title='Predicted',
    yaxis_title='True',
    xaxis=dict(tickmode='array', tickvals=[0, 1], ticktext=['Negative', 'Positive']),
    yaxis=dict(tickmode='array', tickvals=[0, 1], ticktext=['Negative', 'Positive']),
    template='plotly_white'
)

fig_cm.show()

# Plot training history using Plotly (Accuracy and Loss)
fig_history = make_subplots(rows=1, cols=2, subplot_titles=['Model Accuracy', 'Model Loss'])

# Accuracy plot
fig_history.add_trace(go.Scatter(
    x=np.arange(1, epochs+1),
    y=history.history['accuracy'],
    mode='lines',
    name='Train Accuracy'
), row=1, col=1)

fig_history.add_trace(go.Scatter(
    x=np.arange(1, epochs+1),
    y=history.history['val_accuracy'],
    mode='lines',
    name='Val Accuracy'
), row=1, col=1)

# Loss plot
fig_history.add_trace(go.Scatter(
    x=np.arange(1, epochs+1),
    y=history.history['loss'],
    mode='lines',
    name='Train Loss'
), row=1, col=2)

fig_history.add_trace(go.Scatter(
    x=np.arange(1, epochs+1),
    y=history.history['val_loss'],
    mode='lines',
    name='Val Loss'
), row=1, col=2)

# Update layout
fig_history.update_layout(
    title='Training History',
    showlegend=True,
    template='plotly_white',
    width=1100,
    height=600
)

fig_history.show()


# <font color='red'>Save the model and Load the model </font>

In [ ]:
# Save the model to a file
model_save_path = 'classification_model.h5'
classification_model.save(model_save_path)
print(f'Model saved to {model_save_path}')

In [ ]:
from tensorflow.keras.models import load_model

# Load the model from the saved file
loaded_model = load_model(model_save_path)
print('Model loaded successfully!')

# <font color='green'>Ensemble Methods</font>

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier  # Import XGBoost

# Define base models
base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ('xgb', XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=42))
]

# Define meta-model (Logistic Regression)
meta_model = LogisticRegression()

# Create the stacking ensemble
stacking_model = StackingClassifier(estimators=base_models, final_estimator=meta_model, cv=5)

# Train the stacking model
stacking_model.fit(X_train_scaled, y_train)

# Evaluate the model
stacking_accuracy = stacking_model.score(X_test_scaled, y_test)
print(f'Stacking Model Accuracy (with XGBoost): {stacking_accuracy:.4f}')


# <font color='blue'>Train and Evaluate Individual Models </font>

In [ ]:

# Dictionary to store models and results
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=42),
    'Stacking Ensemble': stacking_model  
}

# Function to evaluate a model
def evaluate_model(name, model, X_train_scaled, y_train, X_test_scaled, y_test):
    if name != 'Stacking Ensemble':
        model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else y_pred

    results = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'ROC AUC': roc_auc_score(y_test, y_proba)
    }
    return results


# <font color='red'>Train and Evaluate Individual Models </font>

In [ ]:
# Evaluate and collect results
results_df = pd.DataFrame()

for name, model in models.items():
    results = evaluate_model(name, model, X_train_scaled, y_train, X_test_scaled, y_test)
    results_df[name] = pd.Series(results)

# Transpose for readability
results_df = results_df.T
results_df = results_df.sort_values(by='F1 Score', ascending=False)

# Display the results
print(results_df)


## <font color='#00ffff'>Visualize Model Comparison : 'Accuracy', 'F1 Score', 'ROC AUC', 'Recall', 'Precision'</font>

In [ ]:

# Reset index to make model names a column
results_df_plot = results_df.reset_index().rename(columns={'index': 'Model'})

# Melt the DataFrame to long format for Plotly
results_long = results_df_plot.melt(
    id_vars='Model',
    value_vars=['Accuracy', 'F1 Score', 'ROC AUC', 'Recall', 'Precision'],
    var_name='Metric',
    value_name='Score'
)

# Create interactive grouped bar chart
fig = px.bar(
    results_long,
    x='Model',
    y='Score',
    color='Metric',
    barmode='group',
    text=results_long['Score'].round(2),
    title='Model Comparison: Accuracy, F1 Score, ROC AUC, Recall, Precision',
    labels={'Score': 'Metric Score'},
    height=600,
    width=1000,
    template='plotly_white'
)

fig.update_traces(textposition='outside')

fig.update_layout(
    xaxis_title='Model',
    yaxis_title='Score',
    legend_title='Metric',
    uniformtext_minsize=8,
    uniformtext_mode='hide'
)

fig.show()


# <font color='whiteblack'>Save and Load the Emsemble Model </font>

In [ ]:


# Save the model
joblib.dump(stacking_model, 'stacking_model.pkl')

# load the model:
# loaded_model = joblib.load('stacking_model.pkl')


# <font color='blue'>Thank you! </font>